# MovieLens データセットを使用したハイブリッド レコメンデーション

__注:__ この __als_bqml_hybrid.ipynb__ ノートブックに進む前に、関連する __als_bqml.ipynb__ ノートブックを完了しておくことをお勧めします。ただし、データセットとトレーニング済みモデルを移行するオプションがあるため、これは必須ではありません。すでに movielens データセットとトレーニング済みモデルをお持ちの場合は、「データセットとトレーニング済みモデルのインポート」セクションをスキップできます。

## 学習目標
1. BigQuery 行列分解モデルからユーザー ファクターとプロダクト ファクターを抽出する方法を理解する
2. BigQuery ハイブリッド レコメンデーション モデルの入力をフォーマットする方法を理解する

In [ ]:
PROJECT = !(gcloud config get-value core/project)
PROJECT = PROJECT[0]

%env PROJECT=$PROJECT

%load_ext google.cloud.bigquery

## データセットとトレーニング済みモデルのインポート
前のノートブックでは、2,000 万件の映画レコメンデーション データをインポートし、BigQuery ML で ALS モデルをトレーニングしました。

同じテーブルを使用しますが、これが新しい環境である場合は、以下のコマンドを実行してクリーンなデータをコピーしてください。

まず BigQuery データセットを作成し、データをコピーします。

In [ ]:
!bq mk movielens

In [ ]:
%%bash
rm -r bqml_data
mkdir bqml_data
cd bqml_data
curl -O 'http://files.grouplens.org/datasets/movielens/ml-20m.zip'
unzip ml-20m.zip
yes | bq rm -r $PROJECT:movielens
bq --location=US mk --dataset \
    --description 'Movie Recommendations' \
    $PROJECT:movielens
bq --location=US load --source_format=CSV \
    --autodetect movielens.ratings ml-20m/ratings.csv
bq --location=US load  --source_format=CSV \
    --autodetect movielens.movies_raw ml-20m/movies.csv

そして、クリーンアップされた `movielens.movies` テーブルを作成します。

In [ ]:
%%bigquery --project $PROJECT
CREATE OR REPLACE TABLE movielens.movies AS
    SELECT * REPLACE(SPLIT(genres, "|") AS genres)
    FROM movielens.movies_raw

次に、トレーニング済みのレコメンデーション モデルをコピーします。プロジェクトが EU にある場合は、以下でロケーションを US から EU に変更する必要があることに注意してください。執筆時点では、`bq cp` を使用してリージョン間でモデルをコピーすることはできません。

In [ ]:
%%bash
bq --location=US cp \
cloud-training-demos:movielens.recommender \
movielens.recommender

次に、映画のレコメンデーション予測を呼び出して、モデルが引き続き動作することを確認します。

In [ ]:
%%bigquery --project $PROJECT
SELECT * FROM
ML.PREDICT(MODEL `movielens.recommender`, (
  SELECT 
    movieId, title, 903 AS userId
  FROM movielens.movies, UNNEST(genres) g
  WHERE g = 'Comedy'
))
ORDER BY predicted_rating DESC
LIMIT 5

### ユーザー情報と映画情報の組み込み
行列分解アプローチでは、評価（レーティング）行列から得られる情報以外に、ユーザーや映画に関する情報を一切使用しません。しかし、多くの場合、ユーザー情報（居住都市、年収、年間支出など）が存在し、カタログ内の製品に関する情報もほぼ確実に存在します。これらの情報をレコメンデーション モデルにどのように組み込めばよいでしょうか。

その答えは、行列分解アプローチから得られるユーザー ファクターとプロダクト ファクターが、評価行列から利用可能なユーザーとプロダクトに関する情報の簡潔な表現になっていることを認識することにあります。この情報を利用可能な他の情報と連結し、評価を予測する回帰モデルをトレーニングできます。
### ユーザー ファクターとプロダクト ファクターの取得
ML.WEIGHTS からユーザー ファクターやプロダクト ファクターを取得できます。たとえば、movieId=96481 のプロダクト ファクターと userId=54192 のユーザー ファクターを取得するには、次のように実行します。

In [ ]:
%%bigquery --project $PROJECT
SELECT 
    processed_input,
    feature,
    TO_JSON_STRING(factor_weights) AS factor_weights,
    intercept
FROM ML.WEIGHTS(MODEL `movielens.recommender`)
WHERE
    (processed_input = 'movieId' AND feature = '96481')
    OR (processed_input = 'userId' AND feature = '54192')

これらの重みを乗算し、切片（intercept）を加算することで、行列分解アプローチにおいてこの movieId と userId の組み合わせに対する予測評価が得られます。

また、これらの重みは映画とユーザーの行動の低次元表現としても機能します。ユーザー ファクター、プロダクト ファクター、およびユーザーとプロダクトに関して判明しているその他の情報を入力として、評価を予測する回帰モデルを作成できます。
### 入力特徴量の作成
MovieLens データセットにはユーザー情報がなく、映画自体の情報もごくわずかしかありません。そのため、概念を説明するために、ユーザーに関する合成データを作成してみましょう。

In [ ]:
%%bigquery --project $PROJECT
CREATE OR REPLACE TABLE movielens.users AS
SELECT
    userId,
    RAND() * COUNT(rating) AS loyalty,
    CONCAT(SUBSTR(CAST(userId AS STRING), 0, 2)) AS postcode
FROM
  movielens.ratings
GROUP BY userId

ユーザーに関する入力特徴量は、ユーザー テーブルと ML の重みを結合し、すべてのユーザー情報と重み配列からユーザー ファクターを選択することで取得できます。

In [ ]:
%%bigquery --project $PROJECT
WITH userFeatures AS (
  SELECT 
     u.*,
     (SELECT ARRAY_AGG(weight) FROM UNNEST(factor_weights)) AS user_factors
  FROM movielens.users u
  JOIN ML.WEIGHTS(MODEL movielens.recommender) w
  ON processed_input = 'userId' AND feature = CAST(u.userId AS STRING)
)

SELECT * FROM userFeatures
LIMIT 5

同様に、映画データのプロダクト特徴量も取得できます。ただし、1 つの映画に複数のジャンルが存在する可能性があるため、ジャンルの扱い方を決定する必要があります。ジャンルごとに個別のトレーニング行を作成することにした場合、次のようにプロダクト特徴量を構築できます。

In [ ]:
%%bigquery --project $PROJECT
WITH productFeatures AS (
  SELECT 
      p.* EXCEPT(genres),
      g, (SELECT ARRAY_AGG(weight) FROM UNNEST(factor_weights))
            AS product_factors
  FROM movielens.movies p, UNNEST(genres) g
  JOIN ML.WEIGHTS(MODEL movielens.recommender) w
  ON processed_input = 'movieId' AND feature = CAST(p.movieId AS STRING)
)

SELECT * FROM productFeatures
LIMIT 5

これら 2 つの WITH 句を結合し、（評価テーブルに存在する場合）movieId と userId の組み合わせに対応する評価を取得することで、トレーニング データセットを作成できます。

**TODO 1**: 上記の 2 つのクエリを結合して、各評価に対するユーザー ファクターとプロダクト ファクターを取得します。

In [ ]:
%%bigquery --project $PROJECT
CREATE OR REPLACE TABLE movielens.hybrid_dataset AS

    WITH userFeatures AS (
      SELECT 
         u.*,
         (SELECT ARRAY_AGG(weight) FROM UNNEST(factor_weights))
            AS user_factors
      FROM movielens.users u
      JOIN ML.WEIGHTS(MODEL movielens.recommender) w
      ON processed_input = 'userId' AND feature = CAST(u.userId AS STRING)
    ),

    productFeatures AS (
      SELECT 
          p.* EXCEPT(genres),
          g, (SELECT ARRAY_AGG(weight) FROM UNNEST(factor_weights))
            AS product_factors
      FROM movielens.movies p, UNNEST(genres) g
      JOIN ML.WEIGHTS(MODEL movielens.recommender) w
      ON processed_input = 'movieId' AND feature = CAST(p.movieId AS STRING)
    )

    SELECT
        p.* EXCEPT(movieId),
        u.* EXCEPT(userId),
        rating 
    FROM productFeatures p, userFeatures u
    JOIN movielens.ratings r
    ON r.movieId = p.movieId AND r.userId = u.userId

このテーブルの 1 行は次のようになります。

In [ ]:
%%bigquery --project $PROJECT
SELECT *
FROM movielens.hybrid_dataset
LIMIT 1

基本的には、映画に関するいくつかの属性、映画に対応するプロダクト ファクター配列、ユーザーに関するいくつかの属性、そしてユーザーに対応するユーザー ファクター配列があります。これらは、行列分解モデルをベースにし、ユーザーや映画に関するメタデータを追加した「ハイブリッド」レコメンデーション モデルへの入力を構成します。
### ハイブリッド レコメンデーション モデルのトレーニング
執筆時点では、BigQuery ML は回帰モデルへの入力として配列を処理できません。そのため、配列要素をフィールドとする STRUCT（構造体）に配列を変換する関数を定義しましょう。

In [ ]:
%%bigquery --project $PROJECT
CREATE OR REPLACE FUNCTION movielens.arr_to_input_16_users(u ARRAY<FLOAT64>)
RETURNS 
    STRUCT<
        u1 FLOAT64,
        u2 FLOAT64,
        u3 FLOAT64,
        u4 FLOAT64,
        u5 FLOAT64,
        u6 FLOAT64,
        u7 FLOAT64,
        u8 FLOAT64,
        u9 FLOAT64,
        u10 FLOAT64,
        u11 FLOAT64,
        u12 FLOAT64,
        u13 FLOAT64,
        u14 FLOAT64,
        u15 FLOAT64,
        u16 FLOAT64
    > AS (STRUCT(
        u[OFFSET(0)],
        u[OFFSET(1)],
        u[OFFSET(2)],
        u[OFFSET(3)],
        u[OFFSET(4)],
        u[OFFSET(5)],
        u[OFFSET(6)],
        u[OFFSET(7)],
        u[OFFSET(8)],
        u[OFFSET(9)],
        u[OFFSET(10)],
        u[OFFSET(11)],
        u[OFFSET(12)],
        u[OFFSET(13)],
        u[OFFSET(14)],
        u[OFFSET(15)]
));

実行結果は次のようになります。

In [ ]:
%%bigquery --project $PROJECT
SELECT movielens.arr_to_input_16_users(u).*
FROM (SELECT
    [0., 1., 2., 3., 4., 5., 6., 7., 8., 9., 10., 11., 12., 13., 14., 15.] AS u)

プロダクト ファクター配列を名前付きの列に変換する、`movielens.arr_to_input_16_products` という同様の関数を作成できます。

**TODO 2**: サイズ 16 のプロダクト ファクター配列から名前付きの列を返す関数を作成します。

In [ ]:
%%bigquery --project $PROJECT
CREATE OR REPLACE FUNCTION movielens.arr_to_input_16_products(p ARRAY<FLOAT64>)
RETURNS 
    STRUCT<
        p1 FLOAT64,
        p2 FLOAT64,
        p3 FLOAT64,
        p4 FLOAT64,
        p5 FLOAT64,
        p6 FLOAT64,
        p7 FLOAT64,
        p8 FLOAT64,
        p9 FLOAT64,
        p10 FLOAT64,
        p11 FLOAT64,
        p12 FLOAT64,
        p13 FLOAT64,
        p14 FLOAT64,
        p15 FLOAT64,
        p16 FLOAT64
    > AS (STRUCT(
        p[OFFSET(0)],
        p[OFFSET(1)],
        p[OFFSET(2)],
        p[OFFSET(3)],
        p[OFFSET(4)],
        p[OFFSET(5)],
        p[OFFSET(6)],
        p[OFFSET(7)],
        p[OFFSET(8)],
        p[OFFSET(9)],
        p[OFFSET(10)],
        p[OFFSET(11)],
        p[OFFSET(12)],
        p[OFFSET(13)],
        p[OFFSET(14)],
        p[OFFSET(15)]
));

次に、ユーザーおよびプロダクトに関するメタデータと、行列分解アプローチから得られたユーザー ファクターおよびプロダクト ファクターを結び付け、評価を予測する回帰モデルを作成します。

In [ ]:
%%bigquery --project $PROJECT
CREATE OR REPLACE MODEL movielens.recommender_hybrid 
OPTIONS(model_type='linear_reg', input_label_cols=['rating'])
AS

SELECT
  * EXCEPT(user_factors, product_factors),
    movielens.arr_to_input_16_users(user_factors).*,
    movielens.arr_to_input_16_products(product_factors).*
FROM
  movielens.hybrid_dataset

トレーニング データセットの作成に使用したユーザー情報は偽物（loyalty 列の作成における RAND() など）であるため、このモデルの評価指標を確認しても意味はありません。この演習は、どのように実行できるかを実証するために行いました。もちろん、より高度なモデルが必要な場合は、`dnn_regressor` モデルをトレーニングしてハイパーパラメータを最適化することもできます。しかし、そこまで行うのであれば、[AutoML Tables](https://cloud.google.com/automl-tables) の利用を検討した方がよいかもしれません。

Copyright 2022 Google Inc. Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at http://www.apache.org/licenses/LICENSE-2.0 Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License.